# Silver — ERP Customer Location
Country per customer from the ERP.

`bronze.erp_loc_a101` → `silver.erp_customer_location`

## Init

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType, DateType

CATALOG = "workspace"

## Read bronze table

In [ ]:
df = spark.table(f"{CATALOG}.bronze.erp_loc_a101")

## Transformations

### Trim all string columns

In [ ]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### Clean customer id
Ids arrive as `AW-00011000`; remove the dash to match the CRM format.

In [ ]:
df = df.withColumn("cid", F.regexp_replace(col("cid"), "-", ""))

### Normalize country

In [ ]:
df = df.withColumn(
    "cntry",
    F.when(col("cntry") == "DE", "Germany")
     .when(col("cntry").isin("US", "USA"), "United States")
     .when((col("cntry") == "") | col("cntry").isNull(), "n/a")
     .otherwise(col("cntry"))
)

### Rename to business-friendly names

In [ ]:
RENAME_MAP = {
    "cid": "customer_number",
    "cntry": "country"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity check

In [ ]:
df.limit(10).display()

## Write silver table

In [ ]:
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"{CATALOG}.silver.erp_customer_location")

In [ ]:
%sql
SELECT * FROM workspace.silver.erp_customer_location LIMIT 10;